# L20 - Gridsearch
What should the default value in the sk-learn.svm.SVC class be set to other than 'rbf'?

```python
class sklearn.svm.SVC(
    C=1.0, 
    kernel=’rbf’, 
    degree=3,
    gamma=’auto_deprecated’, 
    coef0=0.0, 
    shrinking=True, 
    probability=False, 
    tol=0.001, 
    cache_size=200, 
    class_weight=None, 
    verbose=False, 
    max_iter=-1, 
    decision_function_shape=’ovr’, 
    random_state=None
  )
```  

A first alternative would be 'linear' such that the class boundary becomes a line. Another alternative is 'poly' for a polynomium.

### Qa Explain GridSearchCV
Review the code in the code cells 1) function setup, 2) the actual grid-search and write a short summary, mainly fouced on cell 2. Focus especially on the lines:
```python
grid_tuned = GridSearchCV(model, tuning_parameters, ..
grid_tuned.fit(X_train, y_train)
..
FullReport(grid_tuned , X_test, y_test, time_gridsearch)
```
And explain how `GridSearchCV` works, how the search parameters set is created along with how the overall search mechanism functions. Finally, what role does the parameter `scoring='f1_micro'` play in the `GridSearchCV` and what does `n_jobs=-1` mean?


Code cell 1) the function setup, start by importing all the necessary packages and modeles. It then sets up a global variable defining the model along with a function to print information about the best model found during a given search, 'SearchReport'. This function furthermore uses a helper function, GetBestModelCtor with the nested function GetParams to create a string describing the model. The function thus returns both a short summary string and the best fitted model. The next function 'ClassificationReport' tests evaluates a given model on a test set. 'FullReport' combines the previous function thereby giving a full report of the best model. 'LoadAndSetupData' loads the dataset and splits it into training and testset, the allowed modes/datasets are in this case iris, minst and moon while the default test size is 0.3. It also updates the global variable to the mode. Finally, the last function 'TryKerasImport' checks whether keras is installed using a boolean statement. Code cell 1) is thus mainly for setting up helper functions.

Code cell 2) the actual grid-search, then uses the helper functions from code cell 1) to train several SVM models using gridsearch, evaluating the best one and print the report. It starts by loading and splitting the data using the 'LoadAndSetupData' function, it then creates a base Support vector classifier model (SVM) and defining the hyperparameters the gridsearch has to use, with 2 kernels and 3 C-values for a total of 6 combinations. It also sets CV=5, which means it uses 5 cross-validations, while VERBOSE=0, prevents it from printing during the gridsearch. The line "grid_tuned = GridSearchCV(model, tuning_parameters, .." then creates the GridsearchCV object with the arguments. It then trains and times the gridsearch. The line "grid_tuned.fit(X_train, y_train)" is the one that actually runs the gridseacrh on the data and "FullReport(grid_tuned , X_test, y_test, time_gridsearch)" then prints the full report of the best model.

`GridSearchCV` is thus a tool that automatically tries different hyperparameter combinations and selects the best one using cross-validation. It works using a base model as an input along with vectors of hyperparameters we want the gridsearch to try, where it then sets up each of the models to best tested. To test them it uses cross-validation. The actual search begines when .fit() is called which begines the testing of the different models.

`scoring='f1_micro'` defines the score the models are judged based on, which in this case is the micr-averaged f1 score of the cross-validation results. Finally, `n_jobs=-1`, tells `GridSearchCV` to use all availabe CPU cores when performing the search, which allows it to try to run several models in parrallel. If this number was 1,2,3 or 4 instead the search would instead use that number of CPU cores.


In [2]:
# TODO: Qa, code review..cell 1) function setup

from time import time
import numpy as np
import sys
sys.path.append("/home/swmal19f26/GITMAL")
from sklearn import svm
from sklearn.linear_model import SGDClassifier

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn import datasets
from libitmal import dataloaders as itmaldataloaders # Needed for load of iris, moon and mnist

currmode="N/A" # GLOBAL var!

def SearchReport(model): 
    
    def GetBestModelCTOR(model, best_params):
        def GetParams(best_params):
            ret_str=""          
            for key in sorted(best_params):
                value = best_params[key]
                temp_str = "'" if str(type(value))=="<class 'str'>" else ""
                if len(ret_str)>0:
                    ret_str += ','
                ret_str += f'{key}={temp_str}{value}{temp_str}'  
            return ret_str          
        try:
            param_str = GetParams(best_params)
            return type(model).__name__ + '(' + param_str + ')' 
        except:
            return "N/A(1)"
        
    print("\nBest model set found on train set:")
    print()
    print(f"\tbest parameters={model.best_params_}")
    print(f"\tbest '{model.scoring}' score={model.best_score_}")
    print(f"\tbest index={model.best_index_}")
    print()
    print(f"Best estimator CTOR:")
    print(f"\t{model.best_estimator_}")
    print()
    try:
        print(f"Grid scores ('{model.scoring}') on development set:")
        means = model.cv_results_['mean_test_score']
        stds  = model.cv_results_['std_test_score']
        i=0
        for mean, std, params in zip(means, stds, model.cv_results_['params']):
            print("\t[%2d]: %0.3f (+/-%0.03f) for %r" % (i, mean, std * 2, params))
            i += 1
    except:
        print("WARNING: the random search do not provide means/stds")
    
    global currmode                
    assert "f1_micro"==str(model.scoring), f"come on, we need to fix the scoring to be able to compare model-fits! Your scoreing={str(model.scoring)}...remember to add scoring='f1_micro' to the search"   
    return f"best: dat={currmode}, score={model.best_score_:0.5f}, model={GetBestModelCTOR(model.estimator,model.best_params_)}", model.best_estimator_ 

def ClassificationReport(model, X_test, y_test, target_names=None):
    assert X_test.shape[0]==y_test.shape[0]
    print("\nDetailed classification report:")
    print("\tThe model is trained on the full development set.")
    print("\tThe scores are computed on the full evaluation set.")
    print()
    y_true, y_pred = y_test, model.predict(X_test)                 
    print(classification_report(y_true, y_pred, target_names=target_names))
    print()
    
def FullReport(model, X_test, y_test, t):
    print(f"SEARCH TIME: {t:0.2f} sec")
    beststr, bestmodel = SearchReport(model)
    ClassificationReport(model, X_test, y_test)    
    print(f"CTOR for best model: {bestmodel}\n")
    print(f"{beststr}\n")
    return beststr, bestmodel
    
def LoadAndSetupData(mode, test_size=0.3):
    assert test_size>=0.0 and test_size<=1.0
    
    def ShapeToString(Z):
        n = Z.ndim
        s = "("
        for i in range(n):
            s += f"{Z.shape[i]:5d}"
            if i+1!=n:
                s += ";"
        return s+")"

    global currmode
    currmode=mode
    print(f"DATA: {currmode}..")
    
    if mode=='moon':
        X, y = itmaldataloaders.MOON_GetDataSet(n_samples=5000, noise=0.2)
        itmaldataloaders.MOON_Plot(X, y)
    elif mode=='mnist':
        X, y = itmaldataloaders.MNIST_GetDataSet(load_mode=0)
        if X.ndim==3:
            X=np.reshape(X, (X.shape[0], -1))
    elif mode=='iris':
        X, y = itmaldataloaders.IRIS_GetDataSet()
    else:
        raise ValueError(f"could not load data for that particular mode='{mode}', only 'moon'/'mnist'/'iris' supported")
        
    print(f'  org. data:  X.shape      ={ShapeToString(X)}, y.shape      ={ShapeToString(y)}')

    assert X.ndim==2
    assert X.shape[0]==y.shape[0]
    assert y.ndim==1 or (y.ndim==2 and y.shape[1]==0)    
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=0, shuffle=True
    )
    
    print(f'  train data: X_train.shape={ShapeToString(X_train)}, y_train.shape={ShapeToString(y_train)}')
    print(f'  test data:  X_test.shape ={ShapeToString(X_test)}, y_test.shape ={ShapeToString(y_test)}')
    print()
    
    return X_train, X_test, y_train, y_test

def TryKerasImport(verbose=True):
    
    kerasok = True
    try:
        import keras as keras_try
    except:
        kerasok = False

    tensorflowkerasok = True
    try:
        import tensorflow.keras as tensorflowkeras_try
    except:
        tensorflowkerasok = False
        
    ok = kerasok or tensorflowkerasok
    
    if not ok and verbose:
        if not kerasok:
            print("WARNING: importing 'keras' failed", file=sys.stderr)
        if not tensorflowkerasok:
            print("WARNING: importing 'tensorflow.keras' failed", file=sys.stderr)

    return ok
    
print(f"OK(function setup" + ("" if TryKerasImport() else ", hope MNIST loads works because it seems you miss the installation of Keras or Tensorflow!") + ")")

OK(function setup)


In [5]:
# TODO: Qa, code review..cell 2) the actual grid-search

# Setup data
X_train, X_test, y_train, y_test = LoadAndSetupData(
    'iris')  # 'iris', 'moon', or 'mnist'

# Setup search parameters
model = svm.SVC(
    gamma=0.001
)  # NOTE: gamma="scale" does not work in older Scikit-learn frameworks,
# FIX:  replace with model = svm.SVC(gamma=0.001)

tuning_parameters = {
    'kernel': ('linear', 'rbf'), 
    'C': [0.1, 1, 10]
}

CV = 5
VERBOSE = 0

# Run GridSearchCV for the model
grid_tuned = GridSearchCV(model,
                          tuning_parameters,
                          cv=CV,
                          scoring='f1_micro',
                          verbose=VERBOSE,
                          n_jobs=-1)

start = time()
grid_tuned.fit(X_train, y_train)
t = time() - start

# Report result
b0, m0 = FullReport(grid_tuned, X_test, y_test, t)
print('OK(grid-search)')

DATA: iris..
  org. data:  X.shape      =(  150;    4), y.shape      =(  150)
  train data: X_train.shape=(  105;    4), y_train.shape=(  105)
  test data:  X_test.shape =(   45;    4), y_test.shape =(   45)

SEARCH TIME: 4.55 sec

Best model set found on train set:

	best parameters={'C': 1, 'kernel': 'linear'}
	best 'f1_micro' score=0.9714285714285715
	best index=2

Best estimator CTOR:
	SVC(C=1, gamma=0.001, kernel='linear')

Grid scores ('f1_micro') on development set:
	[ 0]: 0.962 (+/-0.093) for {'C': 0.1, 'kernel': 'linear'}
	[ 1]: 0.371 (+/-0.038) for {'C': 0.1, 'kernel': 'rbf'}
	[ 2]: 0.971 (+/-0.047) for {'C': 1, 'kernel': 'linear'}
	[ 3]: 0.695 (+/-0.047) for {'C': 1, 'kernel': 'rbf'}
	[ 4]: 0.952 (+/-0.085) for {'C': 10, 'kernel': 'linear'}
	[ 5]: 0.924 (+/-0.097) for {'C': 10, 'kernel': 'rbf'}

Detailed classification report:
	The model is trained on the full development set.
	The scores are computed on the full evaluation set.

              precision    recall  f1-score  

### Qb Hyperparameter Grid Search using an SDG classifier
Replace the `svm.SVC` model with an `SGDClassifier` and a suitable set of the hyperparameters for that model instead.

The code for the `SGDClassifier` is basically the same as code cell 2) above, except the model is changed and different hyperparameters are used for the corresponding model. For the `SGDClassifier` we choose to use the hyperparameters, loss, penalty and alpha. See code cell below implementation. 

Loss controls the type of classifier, in this case either hinge for a linear or log_loss for logistic regression. Penalty controls the regularization, where the common, l1, l2 and elasticnet is used and alpha controls the size of the regularization. 

In [6]:
#Qb - SDG classifier
# Setup data
X_train, X_test, y_train, y_test = LoadAndSetupData(
    'iris')  # 'iris', 'moon', or 'mnist'

# Setup search parameters
model = SGDClassifier(
    random_state=0, max_iter=500, tol=1e-5
) 

tuning_parameters = {
    'loss': ['hinge', 'log_loss'], 
    'penalty': ['l1', 'l2', 'elasticnet'],
    'alpha': [0.0001, 0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008, 0.009, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.2]
}

CV = 5
VERBOSE = 0

# Run GridSearchCV for the model
grid_tuned = GridSearchCV(model,
                          tuning_parameters,
                          cv=CV,
                          scoring='f1_micro',
                          verbose=VERBOSE,
                          n_jobs=-1)

start = time()
grid_tuned.fit(X_train, y_train)
t = time() - start

# Report result
b0, m0 = FullReport(grid_tuned, X_test, y_test, t)

DATA: iris..
  org. data:  X.shape      =(  150;    4), y.shape      =(  150)
  train data: X_train.shape=(  105;    4), y_train.shape=(  105)
  test data:  X_test.shape =(   45;    4), y_test.shape =(   45)

SEARCH TIME: 0.21 sec

Best model set found on train set:

	best parameters={'alpha': 0.006, 'loss': 'hinge', 'penalty': 'l1'}
	best 'f1_micro' score=0.9904761904761905
	best index=36

Best estimator CTOR:
	SGDClassifier(alpha=0.006, max_iter=500, penalty='l1', random_state=0,
              tol=1e-05)

Grid scores ('f1_micro') on development set:
	[ 0]: 0.876 (+/-0.311) for {'alpha': 0.0001, 'loss': 'hinge', 'penalty': 'l1'}
	[ 1]: 0.790 (+/-0.230) for {'alpha': 0.0001, 'loss': 'hinge', 'penalty': 'l2'}
	[ 2]: 0.790 (+/-0.196) for {'alpha': 0.0001, 'loss': 'hinge', 'penalty': 'elasticnet'}
	[ 3]: 0.781 (+/-0.222) for {'alpha': 0.0001, 'loss': 'log_loss', 'penalty': 'l1'}
	[ 4]: 0.695 (+/-0.196) for {'alpha': 0.0001, 'loss': 'log_loss', 'penalty': 'l2'}
	[ 5]: 0.781 (+/-0.293) for 

### Qc Hyperparameter Random  Search using an SDG classifier

Change the code to run a `RandomizedSearchCV` instead. Use these default parameters for the random search.

```python
random_tuned = RandomizedSearchCV(
    model, 
    tuning_parameters, 
    n_iter=20, 
    random_state=42, 
    cv=CV, 
    scoring='f1_micro', 
    verbose=VERBOSE, 
    n_jobs=-1
)
```

Where the two new parameters, `n_iter` and `random_state` are added. Explain the `n_iter` parameter both in code and conceputally. Is the random search best model close to the grid search model?

To change the code to a randomized search instead, the grid_tuned object is changed to the given random_tuned object and random_tuned.fit is then used instead of grid_tuned.fit. See implementation in code cell below.

Regarding the `n_iter` parameter, it determines how many random combinations is attempted. The randomized search works by randomly combining the given parameter and `n_iter` then determines how many combinations is attempted before the best among them is evaluted. In implementation it is here important to consider that if `n_iter` is larger than the number of possible combination, the results will the be the same as for grid_search.

This expected outcome also results from the setup models, with both the parameters and results being exactly the same for large `n_iter`. If the number of possible combinations is increased, by for example massively increasing the amount of alpha parameters, this changes. The alpha value of the randomized search is still rather to close to the gridsearch but both the loss and penalty hyper parameter has changed.

In [7]:
#Qc - Random search
# Setup data
X_train, X_test, y_train, y_test = LoadAndSetupData(
    'iris')  # 'iris', 'moon', or 'mnist'

# Setup search parameters
model = SGDClassifier(
    random_state=0, max_iter=500, tol=1e-5
) 

tuning_parameters = {
    'loss': ['hinge', 'log_loss'], 
    'penalty': ['l1', 'l2', 'elasticnet'],
    'alpha': [0.0001, 0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008, 0.009, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.2]
}

CV = 5
VERBOSE = 0

# Run RandomizedSearchCV for the model
random_tuned = RandomizedSearchCV(
    model, 
    tuning_parameters, 
    n_iter=10, 
    random_state=42, 
    cv=CV, 
    scoring='f1_micro', 
    verbose=VERBOSE, 
    n_jobs=-1
)

start = time()
random_tuned.fit(X_train, y_train)
t = time() - start

# Report result
b0, m0 = FullReport(random_tuned, X_test, y_test, t)

DATA: iris..
  org. data:  X.shape      =(  150;    4), y.shape      =(  150)
  train data: X_train.shape=(  105;    4), y_train.shape=(  105)
  test data:  X_test.shape =(   45;    4), y_test.shape =(   45)

SEARCH TIME: 0.04 sec

Best model set found on train set:

	best parameters={'penalty': 'elasticnet', 'loss': 'log_loss', 'alpha': 0.03}
	best 'f1_micro' score=0.9333333333333332
	best index=5

Best estimator CTOR:
	SGDClassifier(alpha=0.03, loss='log_loss', max_iter=500, penalty='elasticnet',
              random_state=0, tol=1e-05)

Grid scores ('f1_micro') on development set:
	[ 0]: 0.895 (+/-0.071) for {'penalty': 'l2', 'loss': 'hinge', 'alpha': 0.03}
	[ 1]: 0.819 (+/-0.265) for {'penalty': 'l2', 'loss': 'hinge', 'alpha': 0.003}
	[ 2]: 0.819 (+/-0.185) for {'penalty': 'elasticnet', 'loss': 'hinge', 'alpha': 0.1}
	[ 3]: 0.914 (+/-0.164) for {'penalty': 'l2', 'loss': 'hinge', 'alpha': 0.02}
	[ 4]: 0.819 (+/-0.203) for {'penalty': 'l2', 'loss': 'log_loss', 'alpha': 0.06}
	[ 5]: 0

### Qd MNIST Search Quest II

Report the progress in scoring choosing different models, hyperparameters to search for the search-quest competition using the MNIST data. Also note how you might need to the preprocess your data.

We start by trying SGDClassifier with the hyperparameters 'loss', 'penalty' and 'alpha' from the previous exercises on the new dataset. This gives the result:

best: dat=mnist, score=0.88382, model=SGDClassifier(alpha=0.06,loss='log_loss',penalty='l2')

Which is quite a low score compared to what we are typically looking for. To increase the score, we first try to preprocess the data a bit, as suggested, by adding a standard scaler. The SGDclassifier is quite sensitive to the scale of the input features, since it is gradient based and some features might therefore dominate if not properly scaled. The standard model is added to the model part of the setup of search parameters as:
```python
model = Pipeline([
    ('scaler', StandardScaler()),
    ('sgd', SGDClassifier(...))
])
```
This was signficantly faster and resulted in:

best: dat=mnist, score=0.90284, model=Pipeline(sgd__alpha=0.003,sgd__loss='hinge',sgd__penalty'=l2)

Which is a better score but we should be able to do even better. We try to add a few more hyperparameters to see whether they have a significant effect. The added hyperparameter are the learnings rates "optimal" and "adaptive" along with the initial learning rate, eta0. This then gives

best: dat=mnist, score=0.91814, model=Pipeline(sgd__alpha=0.003,sgd__eta0=0.01,sgd__learning_rate='adaptive',sgd__loss='hinge',sgd__penalty='elasticnet)

Which is again slightly better. We then try to change the model to a Random Forest Classifier. The chosen hyperparameters to consider for this classifier is the number of tress, 'n_estimators', how complex the trees are allowed to get, 'max_depth', the minimum number of samples for a tree node to split, 'min_sample_split' and finally, how many features each tree is allowed to consider when looking for the best split, 'max features This then results in the following score:

best: dat=mnist, score=0.96716, model=RandomForestClassifier(max_depth=None,max_features='sqrt',min_samples_split=5,n_estimators=30)

Which is a quite good improvement compared to the previous models. This model is also significantly faster than the SGDClassifiers. As a final attempt at an even better score, we try a SVC model with the hyperparameters, 'C', controlling regularization, 'kernel', controlling the decision boundary and 'gamma' controlling how flexible the boundary is. The pipeline is also readded since SVC is sensative to feature scaling. This gives

best: dat=mnist, score=0.93053, model=Pipeline(svc__C=0.1,svc__gamma=0.001,svc__kernel='linear')

Because SVC is a CPU-based method, the computation took a really long time and we had to remove some of the more complicated models, including the rbf and poly kernels, simply because they took way too long to run. Because of this, the above score only represents the linear SVC models but it is still quite a good score, only surpassed by the Random Forest Classfier.

The best score, however, is then the one from the Random Forest Classifier which is the one that will be submitted to search quest competetion.



In [7]:
# Qd - mnist search quest
# Setup data
X_train, X_test, y_train, y_test = LoadAndSetupData(
    'mnist')  # 'iris', 'moon', or 'mnist'

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
# Setup search parameters
model = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC())
])

tuning_parameters = {
    'svc__kernel': ['linear'], 
    'svc__C': [0.1, 1, 10, 100],
    'svc__gamma': [0.001, 0.01, 0.1]
}

CV = 5
VERBOSE = 0

# Run RandomizedSearchCV for the model
random_tuned = RandomizedSearchCV(
    model, 
    tuning_parameters, 
    n_iter=3, 
    random_state=42, 
    cv=CV, 
    scoring='f1_micro', 
    verbose=VERBOSE, 
    n_jobs=-1
)

start = time()
random_tuned.fit(X_train, y_train)
t = time() - start

# Report result
b0, m0 = FullReport(random_tuned, X_test, y_test, t)

DATA: mnist..
  org. data:  X.shape      =(70000;  784), y.shape      =(70000)
  train data: X_train.shape=(49000;  784), y_train.shape=(49000)
  test data:  X_test.shape =(21000;  784), y_test.shape =(21000)

SEARCH TIME: 2260.32 sec

Best model set found on train set:

	best parameters={'svc__kernel': 'linear', 'svc__gamma': 0.001, 'svc__C': 0.1}
	best 'f1_micro' score=0.9305306122448981
	best index=2

Best estimator CTOR:
	Pipeline(steps=[('scaler', StandardScaler()),
                ('svc', SVC(C=0.1, gamma=0.001, kernel='linear'))])

Grid scores ('f1_micro') on development set:
	[ 0]: 0.914 (+/-0.005) for {'svc__kernel': 'linear', 'svc__gamma': 0.01, 'svc__C': 100}
	[ 1]: 0.914 (+/-0.005) for {'svc__kernel': 'linear', 'svc__gamma': 0.001, 'svc__C': 100}
	[ 2]: 0.931 (+/-0.006) for {'svc__kernel': 'linear', 'svc__gamma': 0.001, 'svc__C': 0.1}

Detailed classification report:
	The model is trained on the full development set.
	The scores are computed on the full evaluation set.

   